# Phase 3D — Hybrid ViT Embedding Extraction (v5_hybrid)

## What this notebook does

Extracts a **4873-D hybrid embedding** from the SwinUNETR-v5 (triplet-trained) model:

| Component | Source | Dim | Fixes |
|---|---|---|---|
| ROI Octant pool | layers3 → tumor bbox → 2×2×2 pool | 3072 | Morphology (M1, M4) |
| ROI Region pool | layers3 → WT/TC/ET mask-weighted | 1152 | Sub-region texture |
| **Global avg pool** | layers3 → full image avg, NO crop | **384** | **T1, T8 temporal** |
| **Triplet projection** | emb_head(384→256) from v5 weights | **256** | **M6 patient identity** |
| Volumetric | GT label voxel counts | 9 | Explicit morphology |
| **TOTAL** | | **4873** | |

## Why this works
- **Global pool (384-D)** restores the brain-level context that ViT-Base lost by ROI-cropping
  → Should improve T1 Spearman (0.23→0.35+) and T8 Kendall τ (0.09→0.25+)
- **Triplet head (256-D)** uses the patient-discriminative space that triplet loss trained
  → Should improve M6 patient purity (3.2%→20-40%)

## What to attach as Kaggle inputs
| Dataset | Files needed |
|---|---|
| `brats2024-dataset` | `/kaggle/input/.../BraTS-GLI-*/` (images + labels) |
| `swinunetr-checkpoints` | `swinunetr_v5_best.pth` (triplet-trained checkpoint) |

**GPU**: Required (T4 ×1, ~25 min for 1620 scans)


In [ ]:
import subprocess, sys
pkgs = ["monai[all]", "nibabel", "einops"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)
print("✅ Dependencies ready")


In [ ]:
import os, gc, time, json, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader
from monai.data import Dataset
from monai.networks.nets import SwinUNETR
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    Orientationd, Spacingd, NormalizeIntensityd, CropForegroundd,
    SpatialPadd, ResizeWithPadOrCropd, MapTransform
)
warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Output paths ──
OUTPUT_ROOT = Path("/kaggle/working/phase3d_hybrid")
EMB_DIR     = OUTPUT_ROOT / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

# ── Embedding dimensions (will be confirmed at runtime from checkpoint) ──
PATCH      = (128, 128, 128)
C_FEAT     = 384           # layers3 channel dim for SwinUNETR-Base
PROJ_DIM   = 256           # triplet head output
OCT_DIM    = 8 * C_FEAT    # 3072
REG_DIM    = 3 * C_FEAT    # 1152
GLOB_DIM   = C_FEAT        # 384
TRIP_DIM   = PROJ_DIM      # 256
VOL_DIM    = 9
TOTAL_DIM  = OCT_DIM + REG_DIM + GLOB_DIM + TRIP_DIM + VOL_DIM
print(f"Expected embedding layout:")
print(f"  Octant (ROI):  {OCT_DIM}-D  (8 × {C_FEAT})")
print(f"  Region (ROI):  {REG_DIM}-D  (3 × {C_FEAT})")
print(f"  Global (full): {GLOB_DIM}-D  (1 × {C_FEAT})")
print(f"  Triplet head:  {TRIP_DIM}-D  (emb_head projection)")
print(f"  Volumetric:    {VOL_DIM}-D")
print(f"  TOTAL:         {TOTAL_DIM}-D")
print(f"  (will confirm exact C at runtime from checkpoint)")


In [ ]:
# ═══════════════════════════════════════════════════════
# DATA DISCOVERY — copied from Phase3_A1 (proven working)
# ═══════════════════════════════════════════════════════
import nibabel as nib

SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def setup_nii_gz_symlinks(data_dir):
    count = 0
    for nii_gz in Path(data_dir).rglob('*.nii_gz'):
        real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
        link = SYMLINK_DIR / nii_gz.parent.name / real_name
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            os.symlink(str(nii_gz), str(link))
            count += 1
    return count

DATA_ROOT = Path('/kaggle/input')
for ds_dir in DATA_ROOT.iterdir():
    if not ds_dir.is_dir(): continue
    if list(ds_dir.rglob('*.nii_gz')):
        n = setup_nii_gz_symlinks(ds_dir)
        if n: print(f'  Created {n} symlinks in {ds_dir.name}')

NIFTI_ROOT = None
for search_root in [SYMLINK_DIR, DATA_ROOT]:
    if not search_root.exists(): continue
    for c in search_root.rglob('BraTS-GLI-*'):
        if c.is_dir():
            NIFTI_ROOT = c.parent
            break
    if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    raise RuntimeError('No BraTS-GLI-* folders found — check dataset attachments')
print(f'NIFTI_ROOT: {NIFTI_ROOT}')

# Read split metadata (if available)
split_map, scan_meta = {}, {}
for f in DATA_ROOT.rglob('scan_index.json'):
    si = json.load(open(f))
    for s in si.get('training_scans', []):
        pid, sid = s['patient_id'], s['scan_id']
        split = s.get('split')
        if split: split_map[pid] = split
        scan_meta[sid] = {'patient_id': pid, 'timepoint': s.get('timepoint','100'), 'split': split}
    print(f'  Split metadata: {len(split_map)} patients from scan_index.json')
    break

# Find all scan directories
all_dirs = sorted([d for d in NIFTI_ROOT.iterdir() if d.is_dir() and 'BraTS-GLI' in d.name])
training_scans = []
for d in all_dirs:
    files = {m: list(d.glob(f'*-{m}*')) for m in ['t1n','t1c','t2w','t2f']}
    seg   = list(d.glob('*-seg*'))
    if not (all(files[m] for m in files) and seg): continue
    name = d.name; pid = name.rsplit('-',1)[0]; tp = name.rsplit('-',1)[1] if '-' in name else '100'
    split = scan_meta.get(name, {}).get('split') or split_map.get(pid)
    training_scans.append({
        'scan_id': name, 'patient_id': pid, 'timepoint': tp,
        't1n': str(files['t1n'][0]), 't1c': str(files['t1c'][0]),
        't2w': str(files['t2w'][0]), 't2f': str(files['t2f'][0]),
        'seg': str(seg[0]), 'split': split,
    })
print(f'Total scans: {len(training_scans)}')

# Split into train/val (same logic as A1)
if any(s['split'] for s in training_scans):
    train_scans = [s for s in training_scans if s.get('split')=='train']
    val_scans   = [s for s in training_scans if s.get('split')=='val']
    kt = {s['patient_id'] for s in train_scans}; kv = {s['patient_id'] for s in val_scans}
    for s in training_scans:
        if s.get('split'): continue
        if s['patient_id'] in kt: train_scans.append(s)
        elif s['patient_id'] in kv: val_scans.append(s)
        else: train_scans.append(s)
else:
    from collections import defaultdict
    pts = defaultdict(list)
    for s in training_scans: pts[s['patient_id']].append(s)
    pids = sorted(pts.keys()); n80 = int(0.8*len(pids))
    tp_set = set(pids[:n80]); vp_set = set(pids[n80:])
    train_scans = [s for s in training_scans if s['patient_id'] in tp_set]
    val_scans   = [s for s in training_scans if s['patient_id'] in vp_set]

print(f'Train: {len(train_scans)} | Val: {len(val_scans)}')

# Build MONAI-compatible dicts
def validate_scan(s):
    for k in ['t1n','t1c','t2w','t2f','seg']:
        try:
            with open(s[k], 'rb') as fh:
                if fh.read(2) != b'\x1f\x8b': return False
        except Exception: return False
    return True

def build_dicts(scan_list):
    dicts, bad = [], []
    for s in scan_list:
        if not validate_scan(s):
            bad.append(s['scan_id']); continue
        dicts.append({
            'image': [s['t1n'],s['t1c'],s['t2w'],s['t2f']],
            'label': s['seg'],
            'patient_id': s['patient_id'],
            'timepoint':  s['timepoint'],
        })
    if bad: print(f'  Skipped {len(bad)} corrupted: {bad[:3]}{"..." if len(bad)>3 else ""}')
    return dicts

print('Validating scans (gzip header check)...')
train_dicts = build_dicts(train_scans)
val_dicts   = build_dicts(val_scans)
all_dicts   = train_dicts + val_dicts
print(f'Train dicts: {len(train_dicts)} | Val dicts: {len(val_dicts)} | Total: {len(all_dicts)}')


In [ ]:
# ═══════════════════════════════════════════════════════
# TRANSFORMS — copied from Phase3_A1 val_transforms
# ═══════════════════════════════════════════════════════
import monai.transforms as T

class ConvertToMultiChannelBrats2024(MapTransform):
    """Correct BraTS 2024 Post-Treatment label mapping.
    1=NETC(necrosis), 2=SNFH(edema), 3=ET(enhancing), 4=RC(cavity, excluded)
    Output: [WT, TC, ET] binary channels."""
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img==1)|(img==2)|(img==3),  # WT = NETC+SNFH+ET (no RC)
                (img==1)|(img==3),           # TC = NETC+ET
                img==3,                      # ET = Enhancing Tissue
            ]
            d[key] = (torch.stack(result, 0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, 0).astype(np.float32))
        return d

val_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats2024(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
print('✅ Transforms ready (3-channel: WT/TC/ET, BraTS 2024)')


In [ ]:
# ═══════════════════════════════════════════════════════
# MODEL LOADING — SwinUNETR-Base + Triplet Head
# ═══════════════════════════════════════════════════════

# ── TripletEmbeddingHead (same as Phase 3A Cell 8) ──
class TripletEmbeddingHead(torch.nn.Module):
    """Projects encoder features to 256-D L2-normalised space."""
    def __init__(self, in_channels=384, proj_dim=256):
        super().__init__()
        self.pool = torch.nn.AdaptiveAvgPool3d(1)
        self.proj = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(in_channels, proj_dim),
            torch.nn.LayerNorm(proj_dim),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(proj_dim, proj_dim),
        )
    def forward(self, feat):
        if feat.dim() == 3:  # (B, N, C) sequence format
            feat = feat.permute(0,2,1).unsqueeze(-1).unsqueeze(-1)
        x = self.pool(feat)
        x = self.proj(x)
        return torch.nn.functional.normalize(x, dim=1)

# ── Build base model ──
# MONAI 1.5.x: img_size removed — do NOT pass it as kwarg
model = SwinUNETR(
    in_channels=4, out_channels=3,
    feature_size=48, use_checkpoint=False,
).to(device)

emb_head = TripletEmbeddingHead(in_channels=C_FEAT, proj_dim=PROJ_DIM).to(device)
HAS_TRIPLET_HEAD = False

# ── Find checkpoint ──
# Search order: v5 best → v4 best → any best → any .pth
# v4/v5 checkpoints have the emb_head weights from triplet training
ckpt_search = (
    list(Path("/kaggle/input").rglob("swinunetr_v5_best.pth")) +
    list(Path("/kaggle/input").rglob("swinunetr_best_v5.pth")) +
    list(Path("/kaggle/input").rglob("swinunetr_best_v4.pth")) +
    list(Path("/kaggle/input").rglob("swinunetr_v4_best.pth")) +
    list(Path("/kaggle/input").rglob("swinunetr_best.pth")) +
    list(Path("/kaggle/input").rglob("swinunetr_latest.pth")) +
    [f for f in Path("/kaggle/input").rglob("*.pth") if "nnunet" not in f.name.lower()]
)

if not ckpt_search:
    raise FileNotFoundError(
        "No checkpoint found!\n"
        "Attach a dataset containing swinunetr_v5_best.pth or swinunetr_best.pth")

ckpt_path = ckpt_search[0]
print(f"Loading checkpoint: {ckpt_path}")

ckpt = torch.load(ckpt_path, map_location=device)
# Handle different checkpoint formats
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))

# Separate model vs emb_head weights
model_state = {k.replace("module.", ""): v for k, v in state.items()
               if not k.startswith("emb_head")}
head_state  = {k.replace("emb_head.", ""): v for k, v in state.items()
               if k.startswith("emb_head")}

# Load model
missing, unexpected = model.load_state_dict(model_state, strict=False)
if missing: print(f"  Missing keys: {missing[:5]}")
model.eval()

# Try loading triplet head
if head_state:
    try:
        emb_head.load_state_dict(head_state, strict=True)
        emb_head.eval()
        HAS_TRIPLET_HEAD = True
        print(f"  ✅ Triplet head loaded — 256-D patient-discriminative projection ACTIVE")
    except Exception as e:
        print(f"  ⚠ Triplet head load failed: {e}")
        print("    Will extract 4617-D (no triplet) instead of 4873-D")
else:
    print("  ⚠ No emb_head weights in checkpoint — will extract 4617-D (base model)")

# Print final dim plan
actual_total = OCT_DIM + REG_DIM + GLOB_DIM + (TRIP_DIM if HAS_TRIPLET_HEAD else 0) + VOL_DIM
print(f"\n{'='*50}")
print(f"  Extraction plan: {actual_total}-D")
print(f"  OCT({OCT_DIM}) + REG({REG_DIM}) + GLOB({GLOB_DIM}) + TRIP({'256' if HAS_TRIPLET_HEAD else 'SKIP'}) + VOL({VOL_DIM})")
print(f"  Triplet head: {'YES ✅' if HAS_TRIPLET_HEAD else 'NO (base model only)'}") 
print(f"  Checkpoint: {ckpt_path.name}")
print(f"{'='*50}")


In [ ]:
# ═══════════════════════════════════════════════════════
# HYBRID EXTRACTION — 4873-D (or 4617-D without triplet)
#
# Layout:
#   [0  :3072] Octant spatial pool (ROI-cropped, 8×C=8×384)
#   [3072:4224] Region mask pool (WT/TC/ET within ROI, 3×C=3×384)
#   [4224:4608] Global avg pool (full image, 1×C=1×384)   ← NEW
#   [4608:4864] Triplet projection (emb_head, 256-D L2)   ← NEW
#   [4864:4873] Volumetric (log_vol×3, flags×3, ratios×3)
#
# Why this fixes the ViT weaknesses:
#   - Global pool restores brain-level context (T1=0.23→~0.35, T8=0.09→~0.25)
#   - Triplet proj captures patient identity (M6=3.2%→~20-40%)
#   - Morphology components unchanged (M1, M4 preserved)
# ═══════════════════════════════════════════════════════

def _is_corrupt(exc):
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream',
            'corrupt','truncat','LoadImaged','applying transform']
    e = exc
    while e is not None:
        if any(k in (type(e).__name__+' '+str(e)) for k in SKIP): return True
        e = e.__cause__ or e.__context__
    return False

def safe_emb_iter(loader):
    it, n_skip = iter(loader), 0
    while True:
        try: yield next(it)
        except StopIteration:
            if n_skip: print(f'  Skipped {n_skip} corrupt files total')
            return
        except Exception as e:
            if _is_corrupt(e): n_skip += 1; continue
            raise

def get_wt_bbox(lbl_feat, min_size=2):
    wt = lbl_feat[0]
    mask = (wt > 0.01).nonzero(as_tuple=False)
    if len(mask) < 1: return None
    z_min, y_min, x_min = mask.min(dim=0).values.tolist()
    z_max, y_max, x_max = mask.max(dim=0).values.tolist()
    h, w, d = wt.shape
    z0=max(z_min-1,0); z1=min(z_max+2,h)
    y0=max(y_min-1,0); y1=min(y_max+2,w)
    x0=max(x_min-1,0); x1=min(x_max+2,d)
    if z1-z0 < min_size: z1=min(z0+min_size,h)
    if y1-y0 < min_size: y1=min(y0+min_size,w)
    if x1-x0 < min_size: x1=min(x0+min_size,d)
    return (z0,z1,y0,y1,x0,x1)

def extract_hybrid(model, emb_head, has_triplet):
    model.eval(); emb_head.eval()
    _feats = {}; hooks = []

    # ── Hook layers3[0] ──
    sv = model.swinViT
    target_layer = 'layers3' if hasattr(sv, 'layers3') else 'layers2'
    layer_list = getattr(sv, target_layer)
    tgt = layer_list[0] if hasattr(layer_list, '__getitem__') else layer_list

    def _hk(m, inp, out):
        feat = out[-1] if isinstance(out, (list, tuple)) else out
        _feats['feat'] = feat.detach()

    hooks.append(tgt.register_forward_hook(_hk))
    print(f"  Hook: swinViT.{target_layer}[0]")

    ds = Dataset(all_dicts, val_transforms)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    total = len(all_dicts)

    embs_arr, ids_arr, tps_arr = [], [], []
    bboxes_arr = []
    n_skip = n_empty = 0
    t_start = time.time()

    with torch.no_grad():
        for idx, batch in enumerate(safe_emb_iter(loader)):
            pid = batch['patient_id'][0]
            tp  = int(batch['timepoint'][0]) if 'timepoint' in batch else 0
            try:
                img = batch['image'].to(device)   # (1,4,128,128,128)
                lbl = batch['label'].to(device)   # (1,3,128,128,128)

                _feats.clear()
                # CropForegroundd produces variable sizes — resize to PATCH for model
                img = F.interpolate(img, list(PATCH), mode='trilinear', align_corners=False)
                lbl = F.interpolate(lbl, list(PATCH), mode='nearest')
                _ = model(img)       # full forward pass (preserves attention)

                if 'feat' not in _feats:
                    n_skip += 1; continue

                feat = _feats['feat']             # (1, C, h, w, d) e.g. (1,384,16,16,16)
                C    = feat.shape[1]
                h, w, d = feat.shape[2:]

                # ── Soft-downsample label to feature resolution ──
                lbl_feat = F.adaptive_avg_pool3d(lbl, (h, w, d))  # (1,3,h,w,d)

                # ── GT volumes at original resolution ──
                wt_vol = float(lbl[0,0].sum().item())
                tc_vol = float(lbl[0,1].sum().item())
                et_vol = float(lbl[0,2].sum().item())

                # ── Component A: ROI Octant + Region pool ──
                bbox = get_wt_bbox(lbl_feat[0], min_size=2)
                if bbox is not None:
                    z0,z1,y0,y1,x0,x1 = bbox
                    feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]

                    # Octant: 2×2×2 pool within ROI → 8C
                    oct_p = F.adaptive_avg_pool3d(feat_crop, (2,2,2))  # (1,C,2,2,2)
                    oct_vec = oct_p[0].reshape(C, 8).T.reshape(-1).cpu()  # (8C,)

                    # Region: mask-weighted within ROI → 3C
                    lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]
                    feat_flat = feat_crop[0].reshape(C, -1)  # (C, N)
                    region_vecs = []
                    for ch in range(3):
                        mask = lbl_crop[0, ch].reshape(-1)
                        vol_soft = float(mask.sum())
                        if vol_soft > 0.01:
                            rvec = (feat_flat * mask.unsqueeze(0)).sum(1) / mask.sum()
                        else:
                            rvec = torch.zeros(C, device=device)
                        region_vecs.append(rvec.cpu())
                    bbox_entry = list(bbox)
                else:
                    oct_vec = torch.zeros(8 * C)
                    region_vecs = [torch.zeros(C) for _ in range(3)]
                    bbox_entry = [0,0,0,0,0,0]
                    n_empty += 1

                # ── Component B: Global avg pool (NO ROI crop) ──
                # This is the key fix for T1, T8 — restores global brain context
                glob_avg = F.adaptive_avg_pool3d(feat, (1,1,1))  # (1,C,1,1,1)
                glob_vec = glob_avg[0].reshape(C).cpu()          # (C,) = 384-D

                # ── Component C: Triplet head projection (if available) ──
                if has_triplet:
                    trip_vec = emb_head(feat).squeeze(0).cpu()   # (256,)
                else:
                    trip_vec = None

                # ── Component D: Volumetric (9-D) ──
                log_wt = np.log1p(wt_vol); log_tc = np.log1p(tc_vol); log_et = np.log1p(et_vol)
                has_wt_f = 1.0 if wt_vol > 10 else 0.0
                has_tc_f = 1.0 if tc_vol > 10 else 0.0
                has_et_f = 1.0 if et_vol > 10 else 0.0
                tc_wt = tc_vol / (wt_vol + 1e-6)
                et_wt = et_vol / (wt_vol + 1e-6)
                et_tc = et_vol / (tc_vol + 1e-6)
                vol_vec = torch.tensor([log_wt, log_tc, log_et,
                                        has_wt_f, has_tc_f, has_et_f,
                                        tc_wt, et_wt, et_tc], dtype=torch.float32)

                # ── Concatenate ──
                parts = [oct_vec] + region_vecs + [glob_vec]
                if trip_vec is not None:
                    parts.append(trip_vec)
                parts.append(vol_vec)
                emb = torch.cat(parts).numpy().astype(np.float32)

                embs_arr.append(emb); ids_arr.append(pid); tps_arr.append(tp)
                bboxes_arr.append(bbox_entry)

                if idx == 0:
                    print(f"\n  [FIRST SCAN] {pid}  tp={tp}")
                    print(f"    feat: {tuple(feat.shape)} | emb dim: {len(emb)}")
                    print(f"    oct={len(oct_vec)} + reg={sum(len(v) for v in region_vecs)}"
                          f" + glob={len(glob_vec)}"
                          f" + trip={len(trip_vec) if trip_vec is not None else 'SKIP'}"
                          f" + vol={len(vol_vec)}")
                    print(f"    WT={wt_vol:.0f}v  TC={tc_vol:.0f}v  ET={et_vol:.0f}v")

                if (idx+1) % 100 == 0 or idx == 0:
                    elapsed = time.time() - t_start
                    rate = (idx+1) / max(elapsed, 1e-6)
                    eta = (total-idx-1) / max(rate, 1e-6)
                    print(f"  [{idx+1:4d}/{total}] {pid[:28]:<28} tp={tp}"
                          f"  | {rate:.1f}/s  ETA={eta/60:.1f}m")

                del img, lbl, feat
                if 'feat_crop' in dir(): del feat_crop
                torch.cuda.empty_cache() if device.type == 'cuda' else None

            except Exception as ex:
                n_skip += 1
                if idx < 10: print(f"  [{idx+1}] ERROR {pid}: {str(ex)[:100]}")

    for h in hooks: h.remove()

    arr = np.array(embs_arr)           # (N, D)
    ids = np.array(ids_arr)
    tps_out = np.array(tps_arr)

    print(f"\n{'='*50}")
    print(f"  Extraction complete: {arr.shape}")
    print(f"  Patients: {len(set(ids_arr))} | Skipped: {n_skip} | Empty ROI: {n_empty}")
    dim = arr.shape[1]
    exp_dim = OCT_DIM + REG_DIM + GLOB_DIM + (TRIP_DIM if has_triplet else 0) + VOL_DIM
    print(f"  Dim: {dim}  (expected {exp_dim}{'  ✅' if dim == exp_dim else '  ⚠ MISMATCH'})")

        # ── Save NPZ ──
    fname = "vit_swinunetr_embeddings_v5_hybrid.npz" if has_triplet else "vit_swinunetr_embeddings_v5_noglobal.npz"
    out_path = EMB_DIR / fname
    np.savez_compressed(out_path, embeddings=arr, patient_ids=ids, timepoints=tps_out)
    print(""); print(f"  ✅ Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)")

    # ── Save NPZ (Triplet Only - 256D) ──
    if has_triplet:
        TRIP_START_IDX = 4608 # Octant(3072) + Region(1152) + Global(384)
        arr_triplet = arr[:, TRIP_START_IDX:TRIP_START_IDX+256]
        out_triplet_path = EMB_DIR / "vit_swinunetr_embeddings_v5_triplet_only.npz"
        np.savez_compressed(out_triplet_path, embeddings=arr_triplet, patient_ids=ids, timepoints=tps_out)
        print(f"  ✅ Saved: {out_triplet_path} ({out_triplet_path.stat().st_size/1e6:.1f} MB) - 256D ONLY")
    
# ── Save tumor_volumes.csv (same schema as before) ──
    vol_rows = []
    for i, (p, t, emb_v) in enumerate(zip(ids_arr, tps_arr, embs_arr)):
        v9 = emb_v[-9:]
        wt_v = float(np.expm1(v9[0])); tc_v = float(np.expm1(v9[1])); et_v = float(np.expm1(v9[2]))
        vol_rows.append({
            "patient_id": p, "timepoint": int(t),
            "wt_vol": wt_v, "tc_vol": tc_v, "et_vol": et_v,
            "has_wt": float(v9[3]>0.5), "has_tc": float(v9[4]>0.5), "has_et": float(v9[5]>0.5),
            "tc_wt_ratio": float(v9[6]), "et_wt_ratio": float(v9[7]), "et_tc_ratio": float(v9[8])
        })
    vol_df = pd.DataFrame(vol_rows)
    vol_csv = EMB_DIR / "tumor_volumes_hybrid.csv"
    vol_df.to_csv(vol_csv, index=False)
    print(f"  ✅ Saved: {vol_csv}  ({len(vol_df)} rows)")
    return arr, ids, tps_out

embs_arr, ids_arr, tps_arr = extract_hybrid(model, emb_head, HAS_TRIPLET_HEAD)


In [ ]:
# ═══════════════════════════════════════════════════════
# EMBEDDING STATISTICS — comparable to CNN baseline
# CNN baseline for reference:
#   Norm CV: 0.22 (raw) → 0.047 (after norm)
#   Diversity: 0.586 (raw) → 0.227 (after norm)
#   RankMe: 121.9 (raw) → 824.4 (after component norm!)
# ═══════════════════════════════════════════════════════
arr = embs_arr
D = arr.shape[1]
np.random.seed(42)

# Determine actual component sizes from shape
if HAS_TRIPLET_HEAD:
    C = (D - 9 - PROJ_DIM) // 11   # solve: 8C + 3C + C + 256 + 9 = D → 12C = D - 265
    GLOB_START  = 8*C + 3*C
    TRIP_START  = GLOB_START + C
    VOL_START   = TRIP_START + PROJ_DIM
else:
    C = (D - 9) // 12               # solve: 8C + 3C + C + 9 = D → 12C = D - 9
    GLOB_START  = 8*C + 3*C
    TRIP_START  = None
    VOL_START   = GLOB_START + C

print("="*60)
print("  RAW EMBEDDING STATISTICS")
print("="*60)
norms = np.linalg.norm(arr, axis=1)
print(f"  Shape:   {arr.shape}")
print(f"  Norm:    min={norms.min():.1f}  max={norms.max():.1f}  mean={norms.mean():.1f}")
print(f"  Norm CV: {norms.std()/norms.mean():.4f}  (CNN raw: 0.2215)")

# Cosine sim
idx = np.random.choice(len(arr), min(1000, len(arr)), replace=False)
pairs = [(idx[i], idx[i+1]) for i in range(0, len(idx)-1, 2)]
cs = [np.dot(arr[a], arr[b])/(np.linalg.norm(arr[a])*np.linalg.norm(arr[b])+1e-8)
      for a, b in pairs[:500]]
print(f"  Cosine sim: mean={np.mean(cs):.3f}  (CNN raw: 0.794)")

# Per-component norms
print(f"\n  Component norms (mean):")
for name, sl in [("Octant(ROI)", slice(0, 8*C)),
                  ("Region(ROI)", slice(8*C, 8*C+3*C)),
                  ("Global(full)", slice(GLOB_START, GLOB_START+C)),
                  ("Triplet(256)", slice(TRIP_START, TRIP_START+PROJ_DIM) if TRIP_START else None),
                  ("Volumetric", slice(VOL_START, VOL_START+9))]:
    if sl is None: continue
    comp = arr[:, sl]
    cn = np.linalg.norm(comp, axis=1)
    print(f"    {name:<20s}: mean={cn.mean():.1f}  CV={cn.std()/(cn.mean()+1e-8):.3f}")

print("\n" + "="*60)
print("  COMPONENT-WISE L2 NORM (same as evaluation notebook)")
print("="*60)
# Apply component-wise L2 (same as unified eval Cell 1)
C_total = 8*C + 3*C  # octant + region
oct_n = arr[:, :8*C] / (np.linalg.norm(arr[:, :8*C], axis=1, keepdims=True) + 1e-8)
reg_n = arr[:, 8*C:8*C+3*C] / (np.linalg.norm(arr[:, 8*C:8*C+3*C], axis=1, keepdims=True) + 1e-8)
glob_n = arr[:, GLOB_START:GLOB_START+C] / (np.linalg.norm(arr[:, GLOB_START:GLOB_START+C], axis=1, keepdims=True) + 1e-8)
if TRIP_START:
    trip_n = arr[:, TRIP_START:TRIP_START+PROJ_DIM] / (np.linalg.norm(arr[:, TRIP_START:TRIP_START+PROJ_DIM], axis=1, keepdims=True) + 1e-8)
vol_n = arr[:, VOL_START:] / (np.linalg.norm(arr[:, VOL_START:], axis=1, keepdims=True) + 1e-8)

parts = [oct_n, reg_n, glob_n * 2]   # ×2 for global (same weight amplification as vol)
if TRIP_START: parts.append(trip_n * 3)  # ×3 for triplet (boost patient signal)
parts.append(vol_n * 2)
arr_norm = np.concatenate(parts, axis=1)

post_norms = np.linalg.norm(arr_norm, axis=1)
print(f"  Post-norm dim: {arr_norm.shape[1]}")
print(f"  Post-norm: mean={post_norms.mean():.3f}  CV={post_norms.std()/post_norms.mean():.4f}")

# Post-norm cosine
cs_post = [np.dot(arr_norm[a], arr_norm[b])/(np.linalg.norm(arr_norm[a])*np.linalg.norm(arr_norm[b])+1e-8)
           for a, b in pairs[:500] if np.linalg.norm(arr_norm[a])>1e-8]
print(f"  Post-norm cosine: mean={np.mean(cs_post):.3f}  (ViT-Base was 0.879 → target <0.80)")

# Diversity
X_u = arr_norm / (post_norms[:, None] + 1e-8)
sub = X_u[np.random.choice(len(X_u), min(500, len(X_u)), replace=False)]
dd = [np.linalg.norm(sub[i]-sub[j]) for i in range(len(sub)) for j in range(i+1, min(i+20, len(sub)))]
print(f"  Post-norm diversity: {np.mean(dd):.3f}  (ViT-Base was 0.121 → target >0.20)")

# RankMe
svs = np.linalg.svd(X_u[:min(500, len(X_u))], compute_uv=False)
p = svs / svs.sum(); p = p[p>1e-10]
print(f"  RankMe: {float(np.exp(-np.sum(p*np.log(p)))):.1f}  (ViT-Base was 36, CNN is 824)")

print("\n✅ Stats complete")
print(f"\nCOMPARISON TARGETS (CNN baseline from Phase 2):")
print(f"  Cosine sim:  CNN_raw=0.794 → ViT_base_post=0.879 → Hybrid target<0.80")
print(f"  Diversity:   CNN_post=0.227 → ViT_base=0.121 → Hybrid target>0.20")
print(f"  RankMe:      CNN=824 >> ViT_base=36 → Hybrid (global+trip) target>50")


In [ ]:
# ═══════════════════════════════════════════════════════
# DOWNLOAD INSTRUCTIONS
# ═══════════════════════════════════════════════════════
print("="*60)
print("  PHASE 3D EXTRACTION COMPLETE")
print("="*60)

out_npz = EMB_DIR / ("vit_swinunetr_embeddings_v5_hybrid.npz"
                      if HAS_TRIPLET_HEAD else "vit_swinunetr_embeddings_v5_noglobal.npz")
out_csv = EMB_DIR / "tumor_volumes_hybrid.csv"
print(f"\nFiles to download:")
print(f"  1. {out_npz}")
print(f"     → rename to 'vit_swinunetr_embeddings_v5_hybrid.npz' locally")
print(f"     → place in: Phase3/embeddings/")
print(f"  2. {out_csv}")
print(f"     → place in: Phase3/outputs/")

print(f"\nNext steps:")
print(f"  1. Upload v5_hybrid.npz to Kaggle dataset 'embedding-datasets'")
print(f"  2. In unified B1 notebook, add to ViT search list:")
print(f"     'vit_swinunetr_embeddings_v5_hybrid'")
print(f"  3. Re-run unified evaluation — expect:")
print(f"     T1 Spearman: {0.23:.2f} → ~0.35+  (global pool)")
print(f"     T8 Kendall τ: {0.09:.2f} → ~0.25+  (global pool)")
if HAS_TRIPLET_HEAD:
    print(f"     M6 Purity:   {3.2:.1f}% → ~20-40%  (triplet head)")
else:
    print(f"     M6 Purity:   {3.2:.1f}% → no change (no triplet head in checkpoint)")
    print(f"     ⚠ To get M6 improvement, upload swinunetr_v5_best.pth (not v3 ckpt)")

print(f"\n{'='*60}")
print(f"  EMBEDDING LAYOUT (v5_hybrid):")
print(f"  D={arr.shape[1]}  layout:")
print(f"  [    0 : {8*C:4d}] ROI Octant pool   (8×{C}={8*C:4d}-D)")
print(f"  [{8*C:4d} : {8*C+3*C:4d}] ROI Region pool   (3×{C}={3*C:4d}-D)")  
print(f"  [{GLOB_START:4d} : {GLOB_START+C:4d}] Global avg pool   (1×{C}= {C:4d}-D)  ← NEW")
if TRIP_START:
    print(f"  [{TRIP_START:4d} : {TRIP_START+PROJ_DIM:4d}] Triplet projection (256-D)  ← NEW")
print(f"  [{VOL_START:4d} : {VOL_START+9:4d}] Volumetric        (9-D)")
print(f"{'='*60}")
